In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/employee_attrition.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))


In [ ]:
data.info()
data.head()


In [ ]:
print("attrition rate:", round(data["attrition"].mean(), 3))
print(data["attrition"].value_counts())


In [ ]:
data.describe().round(2)


In [ ]:
# ======================
# 2. ОТТОК ПО OVERTIME
# ======================
ot_rate = data.groupby("overtime")["attrition"].mean().sort_values()

plt.figure(figsize=(7, 5))
sns.barplot(
    x=ot_rate.index,
    y=ot_rate.values,
    hue=ot_rate.index,
    palette="rocket",
    legend=False,
)
plt.title("Доля attrition по overtime", fontsize=14, pad=15)
plt.xlabel("Overtime", fontsize=12)
plt.ylabel("Доля attrition", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)

for i, val in enumerate(ot_rate.values):
    plt.text(i, val + 0.01, f"{val:.2f}", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig(FIG / "attrition_eda.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Boxplot дохода по attrition
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=data,
    x="attrition",
    y="monthly_income",
    palette="mako",
    hue="attrition",
    legend=False,
)
plt.title("Monthly income по attrition", fontsize=14, pad=15)
plt.xlabel("attrition", fontsize=12)
plt.ylabel("monthly_income", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "attrition_income_box.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Стаж в компании
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=data,
    x="attrition",
    y="years_at_company",
    palette="viridis",
    hue="attrition",
    legend=False,
)
plt.title("Years at company по attrition", fontsize=14, pad=15)
plt.xlabel("attrition", fontsize=12)
plt.ylabel("years_at_company", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "attrition_years_box.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Job satisfaction vs attrition
sat = data.groupby("job_satisfaction")["attrition"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(x=sat.index.astype(str), y=sat.values, palette="coolwarm", hue=sat.index.astype(str), legend=False)
plt.title("Attrition rate по job_satisfaction", fontsize=14, pad=15)
plt.xlabel("job_satisfaction", fontsize=12)
plt.ylabel("Доля attrition", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "attrition_satisfaction.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Корреляции числовых полей
num = ["age", "years_at_company", "monthly_income", "job_satisfaction", "distance_km", "attrition"]
corr = data[num].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Корреляции: employee attrition", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(FIG / "attrition_corr.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 3. БЫСТРЫЙ BASELINE
# ======================
num_cols = ["age", "years_at_company", "monthly_income", "job_satisfaction", "distance_km"]
cat_cols = ["overtime"]

X = data[num_cols + cat_cols]
y = data["attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)


In [ ]:
pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:, 1]

print("AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred, digits=3))


In [ ]:
# Confusion matrix baseline
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
ax.set_title("Attrition — LogReg confusion")
plt.tight_layout()
plt.savefig(FIG / "attrition_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("EDA: overtime и satisfaction связаны с оттоком.")
print("Baseline LogReg — для ориентира AUC/report.")
